# Your own vision model

Train a detector, export it to ONNX, and run it on `g1lab.mock()` on your laptop; on the
robot once its side is switched on. Your program gets back what it found, each thing with
a 3-D point. This module goes through the whole path on the mock, which checks and runs
your model with the robot's own code (but not the robot's 100 ms limit):

1. A model small enough to read, and a post.py
2. Running it: boxes and 3-D points
3. A trained detector from Ultralytics
4. When an upload is refused
5. On the robot

It builds on [Camera and depth](camera_and_depth.ipynb) (from a pixel to a 3-D point)
and [Perception](perception.ipynb), which finds the same red ball by hand.

It needs `pip install onnx onnxruntime`: onnxruntime runs the model, and onnx builds part
1's. Part 3 also needs `pip install ultralytics`, which brings PyTorch, a large download.

In [ ]:
import os, time, math
import numpy as np
import matplotlib.pyplot as plt
import g1lab

# "mock" runs the robot on this laptop (needs `pip install mujoco`).
# In the lab, the robot computer: HOST = "192.168.123.222"
HOST = os.environ.get("G1LAB_HOST", "mock")
NAME = os.environ.get("G1LAB_NAME", "student")     # your name: it shows on the lab board

r = g1lab.mock(name=NAME, camera="render") if HOST == "mock" else g1lab.connect(HOST, name=NAME)
info = r.info()
print("connected to", HOST, "| session:", info.phase, "| has the robot:", info.owner)

## 1. A model you can read

An ONNX file is a graph of array operations with its numbers built in. This one has a
single operation: for each pixel, red minus green minus blue, each from 0 to 1. A clearly
red pixel scores near 1; grey, white and black score 0 or less. It is Perception's
`find_ball()` test, written as a model.

The input is one picture, `[1, 3, 480, 640]`: one image, three colours, the camera's 480
rows and 640 columns. The output is one score per pixel, `[1, 1, 480, 640]`.

In [ ]:
from onnx import TensorProto, helper, numpy_helper, save

weights = numpy_helper.from_array(np.array([1, -1, -1], np.float32).reshape(1, 3, 1, 1), "W")
graph = helper.make_graph(
    [helper.make_node("Conv", ["images", "W"], ["redness"])],      # red - green - blue, per pixel
    "redness",
    [helper.make_tensor_value_info("images", TensorProto.FLOAT, [1, 3, 480, 640])],
    [helper.make_tensor_value_info("redness", TensorProto.FLOAT, [1, 1, 480, 640])],
    [weights])
model = helper.make_model(graph, opset_imports=[helper.make_opsetid("", 11)])
model.ir_version = 8
save(model, "redness.onnx")
print("saved redness.onnx,", os.path.getsize("redness.onnx"), "bytes")

The model gives a score per pixel, not boxes. A **post.py** turns a model's outputs into
detections. It gets the picture (RGB), the model's outputs as numpy arrays, what was
already decoded (nothing, for this kind of model) and the manifest. It returns a list of
detections, each a box `[x1, y1, x2, y2]` in camera pixels, a `conf` from 0 to 1, and
`cls`, the class's place in the manifest's list.

In [ ]:
POST = """
import numpy as np

def post(image, outputs, detections, manifest):
    red = outputs[0][0, 0] > 0.3                  # the pixels that are clearly red
    v, u = np.nonzero(red)
    if len(u) < 30:                               # too few: no ball
        return []
    return [{"box": [u.min(), v.min(), u.max() + 1, v.max() + 1], "conf": 1.0, "cls": 0}]
"""
with open("post.py", "w") as f:
    f.write(POST)

The **manifest** says what the model is:

- `name`: what you call it, 1 to 40 letters, digits, `-` and `_`.
- `task`: `raw` hands the outputs to post.py. `detect`, `pose` and `segment` read
  Ultralytics' layout themselves (part 3).
- `classes`: the class names, in the model's order.
- `input`: the size the model takes. 640 × 480 is the camera picture's own size, so
  the picture reaches the model as it is. For another size it is shrunk to fit and the
  gap filled with grey.
- `post`: `true` when a post.py comes with it.

`upload()` sends the files, then runs the model on a made-up picture: it must load,
its outputs must fit the manifest, and it is timed. `ms` is that time; on the mock it is
your computer's.

In [ ]:
MANIFEST = {"name": "redness", "task": "raw", "classes": ["ball"],
            "input": {"width": 640, "height": 480}, "post": True}

m = r.vision.upload("redness.onnx", MANIFEST, post="post.py")
print(m.name, m.state, f"{m.ms:.1f} ms a frame")

## 2. Run it

Nothing runs until a program activates the model. Then each camera picture goes through
it, and `r.vision.read()` gives the newest picture's detections.

In [ ]:
r.vision.activate("redness")
time.sleep(0.5)                                   # a few pictures through it

dets = r.vision.read()
for d in dets:
    print(d.model, d.label, f"conf {d.conf:.2f}", "box", np.round(d.box))

t, img = r.camera.read()
plt.figure(figsize=(6, 4.5)); plt.imshow(img)
for d in dets:
    x1, y1, x2, y2 = d.box
    plt.gca().add_patch(plt.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, color="c", lw=2))
    plt.text(x1, y1 - 5, d.label, color="c")
plt.title("what the model found"); plt.show()

Each detection also has `xyz`, its 3-D point in the pelvis frame (x forward, y left, z
up, metres). It comes from depth: the depth points in the middle half of the box, their
median distance, along the line through the box's centre. So it is a point on the surface
facing the camera, not the object's centre. `n_depth` is how many depth points it rests
on, and `xyz` is `None` when there were too few. Check for that before you use it.

With `camera="render"` the mock takes each detection's depth from the picture it draws,
without the real camera's noise. (`r.depth.read()` still raises `NoData` on the mock; this
depth is used only inside `r.vision`.)

In [ ]:
d = dets[0]
print("3-D point (m):", None if d.xyz is None else np.round(d.xyz, 3), "from", d.n_depth, "depth points")

if HOST == "mock" and d.xyz is not None:          # the mock knows where it put the ball
    centre = np.array([0.42, -0.12, 0.92 - 0.79]) # placed in the world; the pelvis stands 0.79 m up
    off = np.linalg.norm(d.xyz - centre)
    print(f"{off / 0.0254:.1f} in from the ball's centre; its radius is {0.04 / 0.0254:.1f} in")

**Try this (on the mock).** Restart the mock with the ball somewhere else, e.g.
`g1lab.mock(name=NAME, camera="render", ball=(0.40, -0.30, 1.05))`, upload and activate
again, and check that the point follows the ball.

## 3. A trained detector

For real objects you train a detector on your own pictures. Camera and depth part 3 and
[Recording a dataset](recording_a_dataset.ipynb) collect them. Ultralytics YOLO trains on
them and exports to ONNX. This part uses a stock model trained on COCO's 80 everyday
classes, one of which is "sports ball". The first run downloads it.

Export at 640 × 640 with Ultralytics' defaults. The manifest's classes come from
`model.names`, and `task` is `detect`: the robot reads YOLO's output itself, so no
post.py is needed.

In [ ]:
try:
    from ultralytics import YOLO
except ImportError:
    YOLO = None
    print("this part needs Ultralytics: pip install ultralytics")

if YOLO is not None:
    yolo = YOLO("yolo11n.pt")
    path = yolo.export(format="onnx", imgsz=640)
    COCO = {"name": "coco", "task": "detect", "classes": [yolo.names[i] for i in range(len(yolo.names))],
            "input": {"width": 640, "height": 640}}
    m = r.vision.upload(path, COCO)
    print(m.name, m.state, f"{m.ms:.1f} ms a frame,", len(m.classes), "classes")

Two models can run at once. Each detection says which model found it, and
`read("coco")` gives one model's alone.

In [ ]:
if YOLO is not None:
    r.vision.activate("coco")
    time.sleep(1.0)
    for d in r.vision.read():
        xyz = "no 3-D point" if d.xyz is None else np.round(d.xyz, 3)
        print(f"{d.model:8s} {d.label:12s} conf {d.conf:.2f}  {xyz}")

`conf` in the manifest (default 0.25) drops weaker detections, and `iou` (default 0.45)
merges overlapping boxes of the same class. A model trained on your own objects in the
lab will find them far more reliably than COCO's classes do.

## 4. When an upload is refused

`upload()` raises `ServiceRefused` with the reason, and a model already stored under that
name stays. Here the manifest says the model takes 640 × 640 when it takes 640 × 480:

In [ ]:
try:
    r.vision.upload("redness.onnx", {**MANIFEST, "name": "wrong", "input": {"width": 640, "height": 640}},
                    post="post.py")
except g1lab.ServiceRefused as e:
    print("refused:", e)

The same goes for a manifest field that is missing or misspelled, a post.py that raises
or returns something that is not a list of detections, and a model slower than the
manifest's optional `max_ms`. A model that fails while it runs stops, and
`r.vision.models()` says why in `stopped`.

## 5. On the robot

On the robot `r.vision` is not switched on yet. Once it is, the same calls work with `HOST`
set to the robot computer, with these differences:

- Models stay on the robot until `r.vision.remove(name)`. A model belongs to the name
  your program connected with, so keep `NAME` the same between runs.
- All running models share 100 ms of each camera picture. `upload()` refuses a model
  slower than that, and `activate()` refuses one that would not fit beside those already
  running; training one model on all your classes is then the way.
- post.py runs in a sandbox with numpy, no network and none of the robot's files. What
  it prints goes to a log on the robot.
- A model stops about 30 s after the last program holding it ends.
- The depth is the real camera's: noisier, with holes on plain or shiny surfaces.

The robot's side is not switched on yet: there `upload()` is refused with "r.vision does
not answer on the robot: it is not switched on there yet, or the robot is off. Test your
model on g1lab.mock(camera="render") meanwhile.".

In [ ]:
r.vision.deactivate("redness")
r.vision.remove("redness")
if YOLO is not None:
    r.vision.deactivate("coco")
    r.vision.remove("coco")
r.close()